In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler, MinMaxScaler

# Load dataset
df = pd.read_csv("Games22csv1.csv")

# Dictionary to store final scaled values
scaled_data = {}

# Convert numeric columns into separate 2D NumPy arrays
for col in df.columns:
    if df[col].dtype in [np.int64, np.float64]:
        globals()[col] = df[col].to_numpy().reshape(-1, 1)

# Initialize scalers
robust_scaler = RobustScaler()
minmax_scaler = MinMaxScaler()

# Apply scaling only to numerical columns
for col in df.columns:
    if df[col].dtype in [np.int64, np.float64]:
        # Apply RobustScaler
        scaled_col_name = f"{col}_robust_scaled"
        globals()[scaled_col_name] = robust_scaler.fit_transform(globals()[col])

        # Apply MinMaxScaler
        final_scaled_col_name = f"{col}_final_scaled"
        globals()[final_scaled_col_name] = minmax_scaler.fit_transform(globals()[scaled_col_name])

        # Store final scaled values in dictionary
        scaled_data[final_scaled_col_name] = globals()[final_scaled_col_name].flatten()

# Include categorical columns (like 'team') in the final DataFrame
for col in df.columns:
    if df[col].dtype == object:
        scaled_data[col] = df[col]

# Convert to DataFrame
final_df = pd.DataFrame(scaled_data)

### 🔥 APPLY WEIGHTED MULTIPLICATION AND SUM 🔥 ###
# Define column weights
column_weights = {
    "FGR_2_final_scaled": 0.18,
    "FGR_3_final_scaled": 0.1,
    "FTR_final_scaled": 0.13,
    "AST_final_scaled": 0.11,
    "largest_lead_final_scaled": 0.1,
    "TOV_final_scaled": 0.09,
    "OREB_final_scaled": 0.11,
    "team_score_final_scaled": 0.12,
    "DREB_final_scaled": 0.06
}

# Store modified rows
modified_values = []

for index, row in final_df.iterrows():
    row_updates = {}  # Dictionary to store updated row values
    weighted_sum = 0  # Variable to store the sum of weighted values

    for col in final_df.columns:
        if col == "team":  # Skip categorical columns
            row_updates[col] = row[col]
            continue

        if col in column_weights:  # Check if column needs to be weighted
            new_value = row[col] * column_weights[col]  # Multiply by weight
            weighted_sum += new_value  # Add to total sum
            row_updates[col] = new_value  # Store updated value
        else:
            row_updates[col] = row[col]  # Keep unchanged for other columns

    # Store weighted sum in a new column
    row_updates["weighted_sum"] = weighted_sum
    modified_values.append(row_updates)  # Append updated row

# Convert modified values back to DataFrame
modified_df = pd.DataFrame(modified_values)

### 🔥 SCALE "weighted_sum" TO 0-100 RANGE 🔥 ###
weighted_sum_array = modified_df["weighted_sum"].to_numpy().reshape(-1, 1)
scaled_weighted_sum = MinMaxScaler(feature_range=(0, 100)).fit_transform(weighted_sum_array)
modified_df["weighted_sum_scaled"] = scaled_weighted_sum.flatten()

# Save final modified DataFrame
final_csv_path = "Games22_Scaled_Weighted_100.csv"
modified_df.to_csv(final_csv_path, index=False)

print(f"Final modified data with weighted sum scaled to 0-100 saved to '{final_csv_path}' successfully!")

# If in Google Colab, download file
try:
    from google.colab import files
    files.download(final_csv_path)
    print("Download started for:", final_csv_path)
except ImportError:
    print("Not running in Google Colab, skipping download.")

Final modified data with weighted sum scaled to 0-100 saved to 'Games22_Scaled_Weighted_100.csv' successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started for: Games22_Scaled_Weighted_100.csv


For Attack RTG